This code is designed to get a NIST csv file for a specie and parse it for e.g. SIESTA.

In [44]:
import pandas as pd
import numpy as np
import re

In [45]:
class NISTParser():
    def __init__(self, filepath: str):
        self.filepath = filepath


    def toPandas(self,sep: str, header: int, print_head: bool) -> None:
        
        df = pd.read_csv(self.filepath, sep=sep, header=header)
        print(f"Dataframe loaded from {self.filepath.split('/')[-1]} has: {df.shape[0]} rows and {df.shape[1]} columns")
        print(f"Columns names are: {list(df.columns.values)}")
        
        if print_head:
            print(df.head(5))
        self.dataframe = df
    
    def selectColumns(self, columns: list) -> None:
        if not hasattr(self, 'dataframe'):
            raise ValueError("Dataframe not loaded. Please run toPandas() method first.")
        
        if not all(col in self.dataframe.columns for col in columns):
            missing_cols = [col for col in columns if col not in self.dataframe.columns]
            raise ValueError(f"The following columns are not in the dataframe: {missing_cols}")
        
        selected_df = self.dataframe[columns]
        self.dataframe = selected_df
        print(f"Selected columns: {columns}")

    def intensityFilter(self, threshold_up: float) -> None:
        if not hasattr(self, 'dataframe'):
            raise ValueError("Dataframe not loaded. Please run toPandas() method first.")
        
        if 'intens' not in self.dataframe.columns:
            raise ValueError("Column 'intens' not found in dataframe.")
        
        self.dataframe["intens"] = (self.dataframe["intens"].astype(str).str.replace(r"[^0-9]", "", regex=True).replace("", "0").astype(int))



        filtered_df = self.dataframe[self.dataframe['intens'] >= np.quantile(self.dataframe['intens'], threshold_up)]
        self.dataframe = filtered_df
        print(f"Filtered dataframe to keep intensities >= {threshold_up}. New shape: {self.dataframe.shape}")
        
    def exportDataframeNumpy(self, output_filepath: str) -> None:
        if not hasattr(self, 'dataframe'):
            raise ValueError("Dataframe not loaded. Please run toPandas() method first.")
        
        exported_array = self.dataframe.to_numpy()
        np.save(output_filepath, exported_array)
        print(f"Dataframe exported to numpy array at {output_filepath}")

        

In [46]:
test = pd.read_csv("./NIST_Atomic-Specie/NeI350-875nmtab.csv", sep='\t', header=0)

In [47]:
# test.head(20)


Neon = NISTParser("./NIST_Atomic-Specie/NeI350-875nmtab.csv")
Neon.toPandas(sep='\t', header=0, print_head=False)
# Neon.selectColumns(['obs_wl_air(nm)'])
# Neon.exportDataframeNumpy("./NIST_Atomic-Specie/Neon.npy")
Neon.intensityFilter(threshold_up=0.9)



Dataframe loaded from NeI350-875nmtab.csv has: 664 rows and 19 columns
Columns names are: ['obs_wl_air(nm)', 'unc_obs_wl', 'ritz_wl_air(nm)', 'unc_ritz_wl', 'intens', 'Aki(s^-1)', 'Acc', 'Ei(cm-1)', 'Ek(cm-1)', 'conf_i', 'term_i', 'J_i', 'conf_k', 'term_k', 'J_k', 'Type', 'tp_ref', 'line_ref', 'Unnamed: 18']
Filtered dataframe to keep intensities >= 0.9. New shape: (76, 19)


In [49]:
Neon.dataframe.head(10)

,obs_wl_air(nm),unc_obs_wl,ritz_wl_air(nm),unc_ritz_wl,intens,Aki(s^-1),Acc,Ei(cm-1),Ek(cm-1),conf_i,term_i,J_i,conf_k,term_k,J_k,Type,tp_ref,line_ref,Unnamed: 18
3,352.04714,0.00005,352.047107,0.000018,10000,7580000.0,C+,135888.7173,164285.8872,2s2.2p5.(2P*<1/2>).3s,2[1/2]*,1,2s2.2p5.(2P*<1/2>).4p,2[1/2],0,NaN,T6976,L3451,NaN
5,359.35263,0.00005,359.352567,0.000018,5000,887000.0,C+,135888.7173,163708.6029,2s2.2p5.(2P*<1/2>).3s,2[1/2]*,1,2s2.2p5.(2P*<1/2>).4p,2[3/2],2,NaN,T6976,L3451,NaN
213,453.77545,0.00004,453.775510,0.000030,10000,816000.0,C+,148257.7898,170288.9415,2s2.2p5.(2P*<3/2>).3p,2[1/2],1,2s2.2p5.(2P*<1/2>).5d,2[3/2]*,2,NaN,T6976,L3451,NaN
295,470.43949,0.00004,470.439480,0.000030,15000,2100000.0,C+,148257.7898,169508.5627,2s2.2p5.(2P*<3/2>).3p,2[1/2],1,2s2.2p5.(2P*<3/2>).5d,2[3/2]*,2,NaN,T6976,L3451,NaN
296,470.88594,0.00020,470.885840,0.000030,12000,3260000.0,C+,148257.7898,169488.4193,2s2.2p5.(2P*<3/2>).3p,2[1/2],1,2s2.2p5.(2P*<3/2>).5d,2[1/2]*,1,NaN,T6976,L4498,NaN
297,471.00650,0.00020,471.006380,0.000030,10000,4090000.0,C+,148257.7898,169482.9862,2s2.2p5.(2P*<3/2>).3p,2[1/2],1,2s2.2p5.(2P*<3/2>).5d,2[1/2]*,0,NaN,T6976,L4498,NaN
299,471.20633,0.00020,471.206250,0.000030,15000,NaN,NaN,149657.0392,170873.2327,2s2.2p5.(2P*<3/2>).3p,2[5/2],3,2s2.2p5.(2P*<3/2>).6d,2[5/2]*,3,NaN,NaN,L4498,NaN
306,471.53440,0.00010,471.534410,0.000030,15000,NaN,NaN,149657.0392,170858.4673,2s2.2p5.(2P*<3/2>).3p,2[5/2],3,2s2.2p5.(2P*<3/2>).6d,2[7/2]*,4,NaN,NaN,L7292,NaN
318,475.27320,0.00004,475.273110,0.000040,5000,NaN,NaN,149824.2215,170858.8729,2s2.2p5.(2P*<3/2>).3p,2[5/2],2,2s2.2p5.(2P*<3/2>).6d,2[7/2]*,3,NaN,NaN,L3451,NaN
326,478.89258,0.00010,478.892490,0.000030,10000,NaN,NaN,149657.0392,170532.7169,2s2.2p5.(2P*<3/2>).3p,2[5/2],3,2s2.2p5.(2P*<3/2>).7s,2[3/2]*,2,NaN,NaN,L7292,NaN
